# 조항 수정 에이전트: 지금까지 만든 흐름 따라가기

작성 2026-10-07 · 대상 코드 `src/regcheck/` · 관련 리포트 `docs/reports/261006-1428-07` ~ `261006-1744-11`

이 노트북은 **새 기능을 만들지 않는다.** 이미 만든 코드를 한 단계씩 불러서, 데이터가 어떻게 바뀌는지 눈으로 확인한다.
셀은 위에서부터 순서대로 실행한다. 한 셀은 한 가지 일만 한다.

```
원문 파일(HWP·HWPX·PDF)
  ① 추출      extract.py     파일 → 문단 줄 목록
  ② 파싱      parse.py       줄 → 본문 조·항·호·목 / 부칙 / 별표 / 별지
  ③ 정규화    normalize.py   원문은 그대로, 비교용 norm을 따로
  ④ 메타데이터 build.py       규정ID·위계·개정일·조문 ID
  ⑤ 인용 추출  citations.py   조문 → 인용 대상, 대상이 실제로 있는지 확인
  ⑥ 정답 대조  eval/          1주차 사람 검토와 비교
  ⑦ 관련 조항 표 relations.py  근거·출처·신뢰도·검증 상태를 붙여 저장, 조회
  ⑧ (선택) 벡터 vectors.py     표현이 다른 같은 내용 찾기
  ⑨ 메타데이터 대조            「제35조에 따른 행동강령책임관」 ↔ 제35조 제목 '교육'
```

## 0. 준비

노트북 위치(`notebooks/`)에서 저장소 루트를 찾고, `src/`를 import 경로에 넣는다.
커널은 프로젝트 가상환경(`.venv`)을 쓴다. 설치: `uv sync --group notebook` (벡터 절은 `--extra vector`도 필요).

In [1]:
from pathlib import Path
import sys, json, collections

In [2]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ROOT

WindowsPath('C:/Users/Admin/Desktop/MINI_CNS')

In [3]:
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

In [4]:
import pandas as pd
pd.set_option("display.max_colwidth", 80)

In [5]:
DATA = ROOT / "data" / "한국벤처투자_내부규정"
OUT = ROOT / "out"

## 1. 원문 파일

규정마다 폴더가 하나다. 폴더 이름 끝의 숫자가 ALIO 규정 번호이고, 이것을 **규정ID**로 쓴다.

In [6]:
folders = sorted(p for p in DATA.iterdir() if p.is_dir())
len(folders)

90

In [7]:
[f.name for f in folders[:5]]

['감사규정_23234',
 '감사지침_23245',
 '개인정보 보호 및 관리지침_23250',
 '개인정보 침해사고 대응 및 복구지침_23368',
 '경영공시 관리지침_23382']

In [8]:
exts = collections.Counter(p.suffix.lower() for f in folders for p in f.iterdir())
exts

Counter({'.hwp': 84, '.hwpx': 3, '.pdf': 3})

예시로 **퇴직금지급규정**을 따라간다. 1주차 F1(「인사규정 제6장」)과 F2(「제5조」) 사례가 들어 있다.

In [9]:
folder = next(f for f in folders if f.name.startswith("퇴직금지급규정"))
folder.name

'퇴직금지급규정_24125'

In [10]:
src_file = next(p for p in folder.iterdir() if p.suffix.lower() in (".hwp", ".hwpx", ".pdf"))
src_file.name

'218539_퇴직금지급규정(2025년도 07월 02일 개정).hwp'

## 2. 추출 (`extract.py`)

HWP는 압축된 레코드 묶음이다. 본문 문단은 `PARA_TEXT` 레코드(UTF-16 글자열)에 들어 있다.

In [11]:
from regcheck import extract as ex

In [12]:
raw_paras = ex.extract_hwp(src_file)
len(raw_paras)

101

In [13]:
raw_paras[:6]

['',
 '                                                                                                                                                                 2-6. 퇴직급지급규정',
 '',
 '',
 '퇴 직 금 지 급 규 정  ',
 '제정 : 2000. 06. 15.']

### 2-1. 왜 제어 문자를 규격대로 처리했나

HWP 문단 안에는 표·머리말 같은 컨트롤이 **8글자 묶음**(코드 + 데이터 6 + 코드)으로 끼어 있다.
제어 코드만 지우면 데이터 6글자가 깨진 한자처럼 남는다. 아래는 그 "단순한 방법"을 일부러 재현한 것이다.

In [14]:
import olefile, zlib, struct, re

def naive_first_paras(path, n=3):
    """제어 문자만 지우는 단순한 방법 (비교용)"""
    ole = olefile.OleFileIO(str(path))
    data = zlib.decompress(ole.openstream("BodyText/Section0").read(), -15)
    out = []
    for tag, body in ex._hwp_records(data):
        if tag == ex.HWPTAG_PARA_TEXT:
            out.append(re.sub(r"[\x00-\x1f]", "", body.decode("utf-16le", "ignore")))
        if len(out) == n:
            break
    return out

In [15]:
naive_first_paras(src_file)

['捤獥汤捯慤桥潴景',
 '                                                                                                                                                                 2-6. 퇴직급지급규정',
 '漠杳']

In [16]:
raw_paras[:3]   # 규격대로 처리한 결과: 깨진 한자가 없다

['',
 '                                                                                                                                                                 2-6. 퇴직급지급규정',
 '']

`clean()`은 빈 줄, PDF 쪽번호(「- 5 -」)를 빼고 줄 목록으로 만든다. `extract()`는 형식(HWP·HWPX·PDF)에 맞는 추출기를 고른 뒤 `clean()`까지 한다.

In [17]:
lines = ex.extract(src_file)
len(lines)

96

In [18]:
lines[:12]

['                                                                                                                                                                 2-6. 퇴직급지급규정',
 '퇴 직 금 지 급 규 정',
 '제정 : 2000. 06. 15.',
 '                                                      개정 : 2001. 06. 26.',
 '2002. 07. 16.',
 '2005. 08. 16.',
 '2010. 11. 04.',
 '2013. 06. 19.',
 '2014. 05. 29.',
 '2014. 09. 01.',
 '2014. 12. 29.',
 '2019. 10. 07.']

## 3. 파싱 (`parse.py`)

줄을 하나씩 읽으며 **지금 문서의 어느 구역인지**(머리 → 본문 → 부칙 → 별표·별지)를 기억하는 상태 기계다.

In [19]:
from regcheck.parse import parse

In [20]:
parsed = parse(lines)

In [21]:
parsed.title, parsed.history[:3]

('퇴 직 금 지 급 규 정',
 ['제정 : 2000. 06. 15.', '개정 : 2001. 06. 26.', '2002. 07. 16.'])

In [22]:
len(parsed.articles), len(parsed.addenda), len(parsed.appendices)

(16, 13, 1)

In [23]:
[a.label for a in parsed.articles]

['제1조',
 '제2조',
 '제3조',
 '제4조',
 '제5조',
 '제6조',
 '제7조',
 '제8조',
 '제8조의2',
 '제9조',
 '제10조',
 '제11조',
 '제12조',
 '제13조',
 '제14조',
 '제15조']

### 3-1. 조문 하나 보기: 제3조 (F1 사례)

In [24]:
art3 = next(a for a in parsed.articles if a.label == "제3조")
art3.title, art3.chapter

('지급사유', '제1장  총칙')

In [25]:
print(art3.text)

제3조(지급사유) ① 퇴직금은 다음 각호의 1에 해당하는 사유가 발생하였을 때 지급한다. <개정 2022.08.09., 2025.07.02.>
  1. 인사규정 제6장이 정한 퇴직
  2. 삭제 <2022.08.09.>
  3. 삭제 <2022.08.09.>
  4. 상임임원(이하 “임원”이라 한다)의 임기만료
  5. 사망으로 인한 퇴직
  6. 직원이 임원으로 임명된 경우(단, 일시이사로 선임되는 경우는 예외로 한다)
  7. 고용계약기간 만료시
  ② 제1항의 규정에 불구하고 회사는 특정시점을 기준으로 근로자퇴직급여보장법 제8조에 의한 퇴직금중간정산을 실시할 수 있다.[2014.05.29. 개정]
  ③ 임원 및 직원의 계속근로기간이 1년 미만인 때에는 퇴직금을 지급하지 아니한다. <개정 2022.08.09.>


항·호·목은 `(항, 호, 목)` 경로로 나뉜다. 없는 단계는 0. `삭제 <…>`인 호는 `deleted=True`.

In [26]:
pd.DataFrame([{"경로": it.path, "삭제": it.deleted, "본문": it.text} for it in art3.items])

,경로,삭제,본문
0,"(1, 0, 0)",False,"퇴직금은 다음 각호의 1에 해당하는 사유가 발생하였을 때 지급한다. <개정 2022.08.09., 2025.07.02.>"
1,"(1, 1, 0)",False,인사규정 제6장이 정한 퇴직
2,"(1, 2, 0)",True,삭제 <2022.08.09.>
3,"(1, 3, 0)",True,삭제 <2022.08.09.>
4,"(1, 4, 0)",False,상임임원(이하 “임원”이라 한다)의 임기만료
5,"(1, 5, 0)",False,사망으로 인한 퇴직
6,"(1, 6, 0)",False,"직원이 임원으로 임명된 경우(단, 일시이사로 선임되는 경우는 예외로 한다)"
7,"(1, 7, 0)",False,고용계약기간 만료시
8,"(2, 0, 0)",False,제1항의 규정에 불구하고 회사는 특정시점을 기준으로 근로자퇴직급여보장법 제8조에 의한 퇴직금중간정산을 실시할 수 있다.[2014.05.2...
9,"(3, 0, 0)",False,임원 및 직원의 계속근로기간이 1년 미만인 때에는 퇴직금을 지급하지 아니한다. <개정 2022.08.09.>


### 3-2. 부칙과 별표

In [27]:
[(ad.date, len(ad.articles)) for ad in parsed.addenda][:5]

[('2000. 6. 15', 1),
 ('2001. 6. 26', 1),
 ('2002. 7. 16', 1),
 ('2005. 6. 24', 2),
 ('2005. 8. 16', 2)]

In [28]:
[(ap.kind, ap.number, ap.title) for ap in parsed.appendices]

[('별표', '1', '<신설 2013.6.19>')]

### 3-3. 상태 기계가 막아 주는 것

부칙에도 「제1조(시행일)」이 있고, 계약서형 별지에도 「제1조(목적)」이 있다.
이걸 본문과 섞으면 "제1조가 여러 개"가 된다. 작은 예시로 확인한다.

In [29]:
sample = """테스트규정
제1조(목적) 목적을 정한다.
제2조(정의) ① 용어는 다음과 같다.
  1. 갑
  2. 삭제 <2021.02.03.>
부      칙 <2020.01.01.>
제1조(시행일) 시행한다.
【별지 제1호 서식】
임대차계약서
제1조(목적) 이 계약은 임대차를 목적으로 한다.""".splitlines()

In [30]:
p = parse(sample)
[a.label for a in p.articles]          # 본문 조는 2개뿐

['제1조', '제2조']

In [31]:
[a.label for a in p.addenda[0].articles]  # 부칙의 제1조는 따로

['제1조']

In [32]:
p.appendices[0].text                     # 계약서의 제1조는 별지 텍스트에 남는다

'임대차계약서\n제1조(목적) 이 계약은 임대차를 목적으로 한다.'

## 4. 정규화 (`normalize.py`)

**원문(`text`)은 절대 바꾸지 않는다.** 검토 화면의 근거와 신·구조문대비표의 "현행" 칸에 그대로 써야 하기 때문이다.
비교·검색용으로만 `norm`을 따로 만든다.

In [33]:
from regcheck.normalize import normalize, amend_marks, compact, index_header

In [34]:
t = art3.items[0].text
t

'퇴직금은 다음 각호의 1에 해당하는 사유가 발생하였을 때 지급한다. <개정 2022.08.09., 2025.07.02.>'

In [35]:
normalize(t)      # 개정 표시 제거, 따옴표·가운뎃점 통일, 공백 정리

'퇴직금은 다음 각호의 1에 해당하는 사유가 발생하였을 때 지급한다.'

In [36]:
amend_marks(t)    # 지운 개정 표시는 따로 보관

['<개정 2022.08.09., 2025.07.02.>']

In [37]:
compact("퇴 직 금 지 급 규 정") == compact("퇴직금지급규정")   # 띄어쓰기 무시 비교

True

HWP 머리말(내규집 목차 번호 + 문서명)은 본문이 아니다. 하지만 다른 규정 이름이 들어간 경우가 있어 지우지 않고 따로 본다.

In [38]:
lines[0], index_header(lines[0])

('                                                                                                                                                                 2-6. 퇴직급지급규정',
 '퇴직급지급규정')

## 5. 메타데이터와 조문 ID (`build.py`)

`build()`는 ②~④를 묶고 규정ID·위계·개정일·출처·조문 ID를 붙여 JSON 한 덩어리로 만든다.

In [39]:
from regcheck.build import build, _sources, hierarchy, revision_date

In [40]:
reg = build(folder, _sources(DATA))
list(reg.keys())

['reg_id',
 'name',
 'hierarchy',
 'revised',
 'file',
 'source_url',
 'title_raw',
 'headers',
 'header_mismatch',
 'history',
 'articles',
 'addenda',
 'appendices',
 'preamble']

In [41]:
{k: reg[k] for k in ("reg_id", "name", "hierarchy", "revised", "headers", "header_mismatch")}

{'reg_id': '24125',
 'name': '퇴직금지급규정',
 'hierarchy': '규정',
 'revised': '2025-07-02',
 'headers': ['퇴직급지급규정'],
 'header_mismatch': ['퇴직급지급규정']}

머리말이 「퇴직**급**지급규정」으로 오타가 나 있어 `header_mismatch`에 걸린다 (원문 표기 오류 후보, 원문은 그대로 둠).

In [42]:
hierarchy("취업규칙"), hierarchy("임직원행동강령"), hierarchy("한국벤처투자 정관")

('규정', '지침', '정관')

조문 ID (임시안): 본문 조 `규정ID:조`, 항·호·목 `규정ID:조-항-호-목`, 부칙 `규정ID:부칙N:조`, 별표·별지 `규정ID:별표1`.

In [43]:
[a["id"] for a in reg["articles"][:6]]

['24125:1', '24125:2', '24125:3', '24125:4', '24125:5', '24125:6']

In [44]:
[it["id"] for it in reg["articles"][2]["items"]]

['24125:3-1-0-0',
 '24125:3-1-1-0',
 '24125:3-1-2-0',
 '24125:3-1-3-0',
 '24125:3-1-4-0',
 '24125:3-1-5-0',
 '24125:3-1-6-0',
 '24125:3-1-7-0',
 '24125:3-2-0-0',
 '24125:3-3-0-0']

## 6. 90건 전체

`scripts/parse_regs.py --all`이 만들어 둔 `out/parsed/*.json`을 읽는다. 없으면 다시 만든다 (1~2분).

In [45]:
REBUILD = False

In [46]:
from regcheck.build import build_all
if REBUILD or not list((OUT / "parsed").glob("[0-9]*.json")):
    build_all(DATA, OUT / "parsed")

In [47]:
regs = {r["reg_id"]: r for r in (json.loads(p.read_text(encoding="utf-8")) for p in (OUT / "parsed").glob("[0-9]*.json"))}
len(regs)

90

In [48]:
summary = pd.DataFrame([{
    "규정": r["name"], "위계": r["hierarchy"], "개정": r["revised"],
    "조": len(r["articles"]), "부칙": len(r["addenda"]),
    "별표": sum(x["kind"] == "별표" for x in r["appendices"]),
    "별지": sum(x["kind"] == "별지" for x in r["appendices"]),
} for r in regs.values()]).sort_values("조", ascending=False)
summary.head(10)

,규정,위계,개정,조,부칙,별표,별지
42,회계규정,규정,2022-03,77,10,0,0
31,정보보안관리지침,지침,2019-09,71,6,4,5
40,해외 파견 업무처리 및 해외사무소 운영지침,지침,2025-10-22,61,20,4,11
17,사옥관리지침,지침,2023-05,57,8,2,1
47,인사규정,규정,2026-09-18,55,55,6,2
80,한국벤처투자 정관,정관,2025-07-11,55,7,0,0
46,취업규칙,규정,2025-06-25,54,31,0,2
11,문서관리규정,규정,2026-08-28,49,14,1,23
9,내부통제운영지침,지침,2026-08-28,46,21,2,8
3,개인정보 보호 및 관리지침,지침,2025-04-17,45,8,1,8


In [49]:
summary["조"].sum(), summary["위계"].value_counts().to_dict()

(np.int64(2049), {'지침': 64, '규정': 24, '정관': 1, '기타': 1})

### 6-1. 전체 청킹 결과 (90건)

청크는 두 가지다.
- **조문 구조**: 조 → 항 → 호 → 목. 인용 점검처럼 "제4조 제6호가 있나"를 볼 때 쓴다 (`out/parsed/`)
- **검색용 청크**: 조의 **항 하나**(호·목 포함)에 「규정명 > 제N조(제목)」을 붙인 것. 벡터·키워드 검색에 쓴다 (`vectors.build_chunks`)

부칙·별지·삭제 조는 검색용 청크에서 뺀다. 조문이 없는 윤리헌장은 전문 1개 + 「하나, …」 항목마다 1개로 만든다 (김가빈 수동 청크와 같은 나눔).

In [50]:
from regcheck.vectors import build_chunks
chunks = build_chunks(list(regs.values()))
len(chunks), len({c["reg_id"] for c in chunks})

(4467, 90)

In [51]:
items_total = sum(len(a["items"]) for r in regs.values() for a in r["articles"])
addenda_total = sum(len(r["addenda"]) for r in regs.values())
apps = collections.Counter(x["kind"] for r in regs.values() for x in r["appendices"])
{"본문 조": int(summary["조"].sum()), "항·호·목": items_total, "부칙": addenda_total,
 "별표": apps["별표"], "별지": apps["별지"], "검색용 청크": len(chunks)}

{'본문 조': 2049, '항·호·목': 7345, '부칙': 714, '별표': 105, '별지': 316, '검색용 청크': 4467}

In [52]:
per_reg = pd.DataFrame(collections.Counter(c["reg_name"] for c in chunks).items(), columns=["규정", "청크"])
per_reg = per_reg.merge(summary[["규정", "조"]], on="규정").sort_values("청크", ascending=False)
per_reg.head(10)

,규정,청크,조
31,정보보안관리지침,217,71
87,준법윤리경영지침,158,45
47,인사규정,139,55
46,취업규칙,134,54
11,문서관리규정,132,49
42,회계규정,132,77
3,개인정보 보호 및 관리지침,127,45
40,해외 파견 업무처리 및 해외사무소 운영지침,122,61
9,내부통제운영지침,114,46
80,한국벤처투자 정관,113,55


In [53]:
per_reg.tail(5)      # 청크가 가장 적은 규정 (윤리헌장 5개 포함)

,규정,청크,조
32,제수수료 및 수당지급 지침,9,5
68,안전사고에 관한 문책 지침,8,6
18,서식관리지침,7,6
39,해외 근무직원 복지후생 운영지침,5,3
84,한국벤처투자 윤리헌장,5,0


In [54]:
print(chunks[0]["id"]); print(chunks[0]["text"])

10770:1#0
직원채용지침 > 제1조(목적)
이 지침은 인사규정 제16조에 의거, 직원 채용에 대한 세부 지침을 정함을 목적으로 한다.


In [55]:
[c["id"] for c in chunks if c["reg_name"] == "한국벤처투자 윤리헌장"]

['52754:전문#0', '52754:항목1#0', '52754:항목2#0', '52754:항목3#0', '52754:항목4#0']

참고: 김가빈 챗봇 DB의 색인 대상은 4,465개(조·항 청크, 부칙·삭제 제외). 같은 나눔 원칙이라 규모가 거의 같다. 정확한 대조는 DB 덤프를 받은 뒤 한다.

담당 21건(폴더 가나다순 「직장 내 괴롭힘 예방 운영지침」~「휴직자 복무관리 지침」)만 따로 표시해 둔다.

In [56]:
from parse_regs import assigned_names
ASSIGNED = set(assigned_names())
len(ASSIGNED)

21

## 7. 인용 추출 (`citations.py`)

조문의 `norm`에서 「제N조」, 「「규정명」 제N조」, 「○○법 제N조」를 뽑아 **엣지**로 만들고, 대상이 실제로 있는지 확인한다.
판정은 하지 않는다. `status`는 구조적 사실만 적는다.

In [57]:
from regcheck.citations import Registry, extract_edges, STATUS

In [58]:
registry = Registry(list(regs.values()))   # 90건 규정명·약칭 사전

In [59]:
pd.Series(STATUS, name="뜻")

ok                                            대상 있음
missing_article                             대상 조 없음
missing_item                              대상 항·호 없음
missing_chapter                             대상 장 없음
missing_appendix                        대상 별표·별지 없음
deleted_target                            대상 조가 삭제됨
title_mismatch              인용에 적은 조문 제목이 실제 제목과 다름
name_mismatch             규정명이 공개 규정과 다름 (유사 규정 있음)
unknown_regulation                    공개 90건에 없는 규정
law_pending                     법령 인용 (법률 DB 연결 대기)
external_rule         외부 법령·고시로 추정 (법률 DB 범위 확인 필요)
unresolved                                 대상 해석 불가
Name: 뜻, dtype: str

### 7-1. F3: 회계규정 제72조 「제128조」

In [60]:
acc = next(r for r in regs.values() if r["name"] == "회계규정")
acc_edges = extract_edges(acc, registry)
len(acc_edges)

4

In [61]:
[e for e in acc_edges if e.raw == "제128조"]

[Edge(src='23949:72-0-0-0', src_kind='본문', raw='제128조', context='대표이사는 매회계년도 경과 후 3개월 이내에 제128조의 규정에 의한 결산서를 주주총회 승인을 거쳐 중소벤처기업진흥공단의 대표', target_type='article', target_doc='', target_ref={'jo': 128, 'br': 0, 'hang': None, 'ho': [], 'ttl': None}, target_reg='23949', target_id='', status='missing_article', note='회계규정에 제128조 없음 (마지막 조 제77조)')]

In [62]:
acc["articles"][-1]["label"]     # 회계규정은 제77조까지

'제77조'

### 7-2. 퇴직금지급규정의 인용 전체

In [63]:
ret_edges = extract_edges(reg, registry)

In [64]:
def edges_df(edges):
    return pd.DataFrame([{"출발": e.src, "표현": e.raw, "대상유형": e.target_type,
                          "대상문서": e.target_doc or "(자기)", "대상ID": e.target_id,
                          "상태": e.status, "설명": e.note} for e in edges])

In [65]:
edges_df(ret_edges)

,출발,표현,대상유형,대상문서,대상ID,상태,설명
0,24125:2-0-3-0,제768조,article,민법,,law_pending,
1,24125:3-1-1-0,제6장,chapter,인사규정,24024:장6,ok,"제6장 휴직 및 복직, 직위해제"
2,24125:3-2-0-0,제8조,article,근로자퇴직급여보장법,,law_pending,
3,24125:6-0-0-0,제39조의2,article,한국벤처투자 정관,,missing_article,한국벤처투자 정관에 제39조의2 없음 (마지막 조 제54조)
4,24125:8-1-0-0,제5조,article,(자기),24125:5,ok,퇴직연금제도
5,24125:8의2-1-0-0,제385조제1항,article,상법,,law_pending,
6,24125:8의2-1-0-0,제5조,article,(자기),24125:5,ok,퇴직연금제도
7,24125:14-1-0-0,제41조,article,인사규정,24024:41,ok,명예퇴직
8,24125:14-2-0-0,별표1,appendix,(자기),24125:별표1,ok,<신설 2013.6.19>
9,24125:15-0-0-0,제42조,article,인사규정,24024:42,ok,조기퇴직


- `24125:3-1-1-0` 「제6장」 → 인사규정 제6장(휴직…)이 **존재**하므로 `ok`. 내용이 퇴직이 아니라 휴직이라는 건 구조 검사로는 모른다 (9절).
- `24125:6-0-0-0` 「제39조의2」 → 정관에 없음 (1주차 C13).
- `24125:8-1-0-0` 「제5조」 → 자기 규정 제5조(퇴직연금)가 존재하므로 `ok` (F2, 9절).

### 7-3. 다른 상태들의 예

In [66]:
emp = next(r for r in regs.values() if r["name"] == "취업규칙")
emp_edges = extract_edges(emp, registry)
edges_df([e for e in emp_edges if e.status == "name_mismatch"])

,출발,표현,대상유형,대상문서,대상ID,상태,설명
0,24010:24-5-0-0,「휴직자의 복무관리지침」,regulation,휴직자의 복무관리지침,,name_mismatch,가장 비슷한 규정: 휴직자 복무관리 지침 (0.95)
1,24010:51-2-0-0,「성희롱 예방운영지침」,regulation,성희롱 예방운영지침,,name_mismatch,가장 비슷한 규정: 성희롱·성폭력 예방 운영지침 (0.82)


In [67]:
exe = next(r for r in regs.values() if r["name"] == "집행간부 운영규정")
edges_df([e for e in extract_edges(exe, registry) if e.status != "law_pending"])

,출발,표현,대상유형,대상문서,대상ID,상태,설명
0,23859:1-0-0-0,제42조의 2,article,한국벤처투자 정관,,missing_article,한국벤처투자 정관에 제42조의2 없음 (마지막 조 제54조)
1,23859:1-0-0-0,제3조의 2,article,직제규정,,missing_article,직제규정에 제3조의2 없음 (마지막 조 제20조)
2,23859:2-2-0-0,제5조,article,(자기),23859:5,ok,보수 및 처우 등
3,23859:3-5-0-0,제5조 제2항,article,(자기),23859:5,ok,보수 및 처우 등
4,23859:4-0-2-0,제 6조(채용제한),article,취업규칙,24010:6,title_mismatch,"인용 제목 「채용제한」, 실제 제6조(채용결격사유)"


「제6조(채용제한)」처럼 인용에 제목을 붙인 경우 실제 제목과 비교한다 (`title_mismatch`).

### 7-4. 약칭 풀이

In [68]:
from regcheck.citations import _aliases
_aliases(next(r for r in regs.values() if r["name"] == "부정청탁 및 금품등 수수의 신고사무 처리지침"))

{'법': '부정청탁 및 금품등 수수의 금지에 관한 법률', '영': '부정청탁 및 금품등 수수의 금지에 관한 법률 시행령'}

In [69]:
list(registry.law_aliases.items())[:5]   # 다른 규정에서 정의한 약칭도 쓴다

[('개보법', '개인정보 보호법'),
 ('민원처리법', '민원처리에 관한 법률'),
 ('금융실명법', '금융실명거래 및 비밀보장에 관한 법률'),
 ('국가계약법 시행령', '당사자로 하는 계약에 관한 법률 시행령'),
 ('공공기록물법', '공공기록물 관리에 관한 법률')]

## 8. 담당 21건 전체 + 1주차 정답 대조

`eval/week1_citations.json`은 1주차 사람 검토(§C)에서 인용이 명시된 사례다.
`structural`은 구조 검사로 잡혀야 하는 것, `semantic`은 대상은 있는데 내용이 달라 의미 검사가 필요한 것.

In [70]:
edges21 = []
for r in regs.values():
    if r["name"] in ASSIGNED:
        for e in extract_edges(r, registry):
            edges21.append({**e.to_dict(), "src_reg": r["reg_id"], "src_name": r["name"]})
len(edges21)

441

In [71]:
pd.Series(collections.Counter(e["status"] for e in edges21)).sort_values(ascending=False)

ok                    244
law_pending           155
unknown_regulation     25
missing_article         5
missing_item            4
external_rule           3
name_mismatch           3
title_mismatch          2
dtype: int64

In [72]:
gold = json.loads((ROOT / "eval" / "week1_citations.json").read_text(encoding="utf-8"))["cases"]
len(gold)

16

In [73]:
PROBLEM = {"missing_article", "missing_item", "missing_chapter", "missing_appendix",
           "deleted_target", "name_mismatch", "unknown_regulation", "title_mismatch"}

In [74]:
rows = []
for c in gold:
    cand = [e for e in edges21 if e["src"] == c["src"] and c["raw"] in e["raw"]]
    hit = [e for e in cand if e["status"] in c.get("status", [])]
    rows.append({"ID": c["id"], "기대": c["expect"], "탐지": bool(hit),
                 "상태": (hit or cand or [{"status": "-"}])[0]["status"]})
week1 = pd.DataFrame(rows)
week1

,ID,기대,탐지,상태
0,C1,structural,True,missing_item
1,C2,semantic,False,ok
2,C3,structural,True,title_mismatch
3,C5,semantic,False,ok
4,C7,structural,True,name_mismatch
5,C8,structural,True,name_mismatch
6,C9,structural,True,missing_item
7,C10a,structural,True,missing_article
8,C10b,structural,True,missing_article
9,C11,structural,True,title_mismatch


In [75]:
s = week1[week1["기대"] == "structural"]
f"구조 검사 대상 {len(s)}건 중 {s['탐지'].sum()}건 탐지"

'구조 검사 대상 11건 중 11건 탐지'

## 9. 관련 조항 표 (`relations.py`)

영향 분석은 **수정할 때마다 찾지 않고, 미리 연결해 둔 표를 조회**한다.
관계마다 근거 문구·출처·신뢰도·검증 상태·원문 버전을 붙인다. ("그래프"는 이 표 + 역방향 조회를 말한다.)

In [76]:
from regcheck.relations import from_edges, Related

In [77]:
edges_all = []
for r in regs.values():
    for e in extract_edges(r, registry):
        edges_all.append({**e.to_dict(), "src_reg": r["reg_id"], "src_name": r["name"]})
len(edges_all)

1852

In [78]:
rels = [x.to_dict() for x in from_edges(edges_all, regs)]
len(rels)

1852

In [79]:
pd.DataFrame(rels)[["kind", "confidence", "verification"]].value_counts()

kind      confidence  verification
법령 인용     확실          자동              586
인용        확실          자동              440
별표·별지 인용  확실          자동              413
문서 인용·위임  확실          자동              147
인용        확실          의심               64
          후보          자동               52
문서 인용·위임  확실          의심               40
          후보          자동               40
별표·별지 인용  확실          의심               29
인용        후보          의심               27
문서 인용·위임  후보          의심                9
법령 인용     후보          자동                5
Name: count, dtype: int64

In [80]:
pd.DataFrame(rels).iloc[0]

id                                                            R00000
src                                                    10770:1-0-0-0
src_article                                                  10770:1
src_kind                                                          본문
target                                                      24024:16
target_type                                                  article
kind                                                              인용
evidence                                                        제16조
context           이 지침은 인사규정 제16조에 의거, 직원 채용에 대한 세부 지침을 정함을 목적으로 한다.
source                                                          rule
confidence                                                        확실
verification                                                      자동
check_status                                                      ok
note                                                            채용방법
src_version                       

### 9-1. 조회: 직제규정 제7조(주요조직)를 고치면 누가 영향을 받나

In [81]:
related = Related(rels)

In [82]:
jikje = next(r for r in regs.values() if r["name"] == "직제규정")
target = f"{jikje['reg_id']}:7"
res = related.lookup(target)
{k: len(v) for k, v in res.items()}

{'인용함': 0, '인용받음': 13, '규정 단위로 인용받음': 12}

In [83]:
pd.DataFrame([{"출발": r["src"], "표현": r["evidence"], "문맥": r["context"][:60]} for r in res["인용받음"]])

,출발,표현,문맥
0,23583:22의3-6-0-0,제7조,또는 업무상의 이유로 위원장이 심의·의결할 수 없을 경우에는 직제규정 제7조 직제 순서에 따라 해당 부서장
1,23623:7-2-0-0,제7조제1항,「직제규정」 제7조제1항에 따른 부서의 장은 업무상 이사회의 심의·의결을 거쳐야 하는 사항이 있
2,23801:7-1-2-1,제7조제1항,전결권자가 부기관장인 경우:「직제규정」제7조제1항 직제순서에 따른 부서의 장
3,24024:3-0-14-0,제7조제1항,"""부서장""이라 함은 「직제규정」 제7조제1항에 따른 부서의 업무를 통할하고 소속직원을 지휘 감독하는 자를 말"
4,24184:3-2-2-0,제7조제1항,제7조제1항 직제순서에 따른 부서의 장
5,24184:3-3-0-0,제7조제1항,불구하고 대표이사는 업무의 특성상 특별히 필요하다고 인정되는 경우에는 제7조제1항에 따른 부서의 장 중에서
6,24184:4-2-0-0,제7조제1항,사내이사는 이사회에서 정하는 바에 따라 제7조제1항에 따른 부서의 장을 겸할 수 있다.
7,24184:8-1-0-0,제7조,"대표이사는 제7조, 제10조에 따른 조직의 직무수행을 위하여 하부조직 등으로 실·팀·연구"
8,24184:17-2-0-0,제7조,제7조와 제8조의 적용에 있어 제1항제2호의 전문계약직의 직위는 수석 전문위원
9,24184:19-1-0-0,제7조,"제7조, 제10조, 제13조2호에 따른 조직의 업무분장에 관하여는 「직제지침」"


### 9-2. 조회만으로는 못 찾는 것: J4 육아휴직 분할 횟수

취업규칙 제43조②(3회)와 인사규정 제30조②(2회)는 **서로 인용하지 않는다.** 그래서 관련 조항 표에는 없다.
아래 결과가 비어 있거나 위임(문서 단위)만 나오는 것을 확인한다. 이 빈칸을 채우는 것이 다음 단계(같은 대상·속성 관계)다.

In [84]:
insa = next(r for r in regs.values() if r["name"] == "인사규정")
res30 = related.lookup(f"{insa['reg_id']}:30")
{k: len(v) for k, v in res30.items()}

{'인용함': 0, '인용받음': 1, '규정 단위로 인용받음': 14}

In [85]:
[(r["src"], r["evidence"]) for r in res30["규정 단위로 인용받음"]][:8]

[('10770:17-1-0-0', '인사규정'),
 ('23623:3-2-10-5', '「인사규정」'),
 ('23792:7-0-0-0', '「인사규정」'),
 ('24010:5-2-0-0', '「인사규정」'),
 ('24010:6-0-0-0', '「인사규정」'),
 ('24010:21-2-0-0', '「인사규정」'),
 ('24010:22-2-0-0', '「인사규정」'),
 ('24010:24-5-0-0', '「인사규정」')]

## 10. (선택) 벡터: 표현이 다른 같은 내용

`uv sync --extra vector`와 `scripts/build_vectors.py`가 필요하다. GPU가 없으면 이 절은 건너뛴다.
벡터는 **영향 판단의 근거가 아니라**, 관계를 미리 만들 때 "표현이 다른 같은 개념"을 놓치지 않게 돕는 보조다.

In [86]:
VEC_DIR = OUT / "vector" / "BAAI__bge-m3"
VEC_DIR.exists()

True

In [87]:
try:
    from regcheck.vectors import Index, load_model, embed
    index = Index(VEC_DIR)
    model = load_model()
    VECTOR_OK = True
except Exception as err:
    VECTOR_OK = False
    print("벡터 절 건너뜀:", err)

C:\Users\Admin\Desktop\MINI_CNS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 37872.04it/s]

In [88]:
def search(q, k=5):
    qv = embed(model, [q])[0]
    return [(round(s, 3), c["reg_name"], c["label"], c["title"]) for s, c in index.search(qv, k=k)]

In [89]:
search("당연해직 사유") if VECTOR_OK else None    # 원문 용어는 '당연면직'

[(0.697, '인사규정', '제37조', '당연면직'),
 (0.693, '해외 파견 업무처리 및 해외사무소 운영지침', '제35조', '해고'),
 (0.647, '인사규정', '제39조', '직권면직'),
 (0.629, '인사규정', '제37조', '당연면직'),
 (0.617, '인사규정', '제34조', '직위해제')]

In [90]:
search("육아휴직을 나누어 사용할 수 있는 횟수") if VECTOR_OK else None   # J4 두 조항이 함께 나오는지

[(0.757, '인사규정', '제30조', '육아휴직'),
 (0.754, '취업규칙', '제43조', '육아휴직'),
 (0.709, '취업규칙', '제44조', '육아기근로시간단축'),
 (0.68, '인사규정', '제31조', '가족돌봄휴직'),
 (0.676, '취업규칙', '제46조', '가족돌봄휴직')]

## 11. 메타데이터 대조: 대상 불일치 후보 (GPU 불필요)

인용 바로 뒤의 **설명구**(「제35조에 따른 **행동강령책임관**」)를 뽑아, 인용된 조문의 **제목**에 그 단어가 있는지 본다.
없으면 "대상 불일치 후보". 제목에 그 단어가 있는 다른 조문을 대안 후보로 보여 준다. 판정은 사람이 한다.

In [91]:
from semantic_check import descriptor

In [92]:
c2 = next(e for e in edges21 if e["src"] == "40209:5-4-0-0" and e["raw"] == "제35조")
c2["target_doc"], c2["target_id"], c2["status"]

('임직원행동강령', '23678:35', 'ok')

In [93]:
desc = descriptor(regs, c2)
desc

'행동강령책임관'

In [94]:
target_reg = regs[c2["target_reg"]]
cited = next(a for a in target_reg["articles"] if a["id"] == c2["target_id"])
cited["label"], cited["title"]

('제35조', '교육')

In [95]:
compact(desc) in compact(cited["title"])     # 제목에 '행동강령책임관'이 없다 → 후보

False

In [96]:
[(a["label"], a["title"]) for a in target_reg["articles"] if compact(desc) in compact(a["title"])]

[('제36조', '행동강령책임관의 지정')]

1주차 수정안도 제36조다. 같은 방식으로 담당 21건의 설명구 있는 인용을 모두 본다.

In [97]:
rows = []
for e in edges21:
    if e["target_type"] != "article" or e["status"] not in ("ok", "title_mismatch") or not e["target_id"]:
        continue
    d = descriptor(regs, e)
    if not d:
        continue
    t = next((a for a in regs[e["target_reg"]]["articles"] if a["id"] == e["target_id"]), None)
    if t:
        rows.append({"출발": e["src"], "인용": e["raw"], "설명구": d, "대상 제목": t["title"],
                     "제목에 있음": compact(d) in compact(t["title"])})
meta = pd.DataFrame(rows)
meta[~meta["제목에 있음"]]

,출발,인용,설명구,대상 제목,제목에 있음
2,23926:11의1-0-0-0,제22조의 2,해외출장심사위원회,해외출장 사전계획수립 및 사전 심사실시,False
3,23926:25-0-0-0,제22조의 2,해외출장심사위원회,해외출장 사전계획수립 및 사전 심사실시,False
5,23926:별표1,제5조,해외파견자 선발 심사위원회,해외파견자의 선발,False
6,24125:8-1-0-0,제5조,퇴직금 이외,퇴직연금제도,False
7,24125:8의2-1-0-0,제5조,퇴직금,퇴직연금제도,False
8,24125:14-1-0-0,제41조,명예퇴직자에게,명예퇴직,False
9,24125:15-0-0-0,제42조,조기퇴직자에게,조기퇴직,False
10,40209:5-4-0-0,제35조,행동강령책임관,교육,False
11,40209:13-2-0-0,제10조,정식 조사기간 동안 피해근로자등,정식 조사,False
12,43061:1-0-0-0,제26조,휴직 기간 중인 자,휴직의 종류,False


단어가 완전히 같지 않아도 겹치는 경우(「퇴직금」 vs 「퇴직연금제도」 등)가 있어, 실제 기준은 두 글자 단위 겹침 비율로 다듬는다 (리포트 261006-1603-10, 대화 기록).

## 12. 정리

| 단계 | 상태 | 이 노트북 절 |
|---|---|---|
| 추출·파싱·정규화·메타데이터 | 90건 완료 | 2~6 |
| 인용 추출·구조 검사 | 1주차 구조 대상 11/11 | 7~8 |
| 관련 조항 표 | 인용·문서·법령·별표 관계, 표본 정확도 측정 중 | 9 |
| 같은 대상·속성(값), 정의-사용, 예외 관계 | **미착수** (9-2의 빈칸) | — |
| 대상 불일치 후보 | 메타데이터 대조 + reranker 보조 | 11 |
| 개정문 적용·신·구조문대비표·검토 화면·법령 연결 | **미착수** | — |

다음에 적용할 때는 9-2의 빈칸(J4)을 채우는 관계부터 붙이고, 같은 노트북 흐름에 절을 추가한다.